# Imports

In [1]:
# Imports and PyRosetta initialization
from pathlib import Path
import os
import re
import sys
import time
import warnings

import numpy as np
import pandas as pd
import pyrosetta
from pyrosetta import rosetta
from scipy.stats import pearsonr, spearmanr

warnings.filterwarnings("ignore")

pyrosetta.init(
    "-mute all "
    "-ignore_unrecognized_res true "
    "-load_PDB_components false "
    "-linmem_ig 10 "
    "-ex1 -ex2aro -use_input_sc"
)

┌───────────────────────────────────────────────────────────────────────────────┐
│                                  PyRosetta-4                                  │
│               Created in JHU by Sergey Lyskov and PyRosetta Team              │
│               (C) Copyright Rosetta Commons Member Institutions               │
│                                                                               │
│ NOTE: USE OF PyRosetta FOR COMMERCIAL PURPOSES REQUIRES PURCHASE OF A LICENSE │
│          See LICENSE.PyRosetta.md or email license@uw.edu for details         │
└───────────────────────────────────────────────────────────────────────────────┘
PyRosetta-4 2025 [Rosetta PyRosetta4.Release.python310.m1 2025.41+release.de3cc17d509259e29147a2ed8f2a726d644e7e34 2025-10-06T16:25:46] retrieved from: http://www.pyrosetta.org


# Path

In [ ]:
# Paths and dataset configuration
DATA_DIR = Path("../../data")
EXPERIMENTAL_DIR = DATA_DIR / "experimental_datasets"
TOURNAMENT_DIR = DATA_DIR / "tournament"
PDB_DIR = EXPERIMENTAL_DIR / "pdb"

OUTPUT_DIR = Path("rosetta_zero_shot_outputs")
MUT_PDB_ROOT = OUTPUT_DIR / "mut_pdbs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MUT_PDB_ROOT.mkdir(parents=True, exist_ok=True)

SCORE_COL = "ddg"
VARIANT_RE = re.compile(r"^([A-Z])(\d+)([A-Z])$")

DATASET_CONFIG = {
    "UBE4B": {
        "data_path": EXPERIMENTAL_DIR / "ube4b_with_full_sequence.tsv",
        "sep": "\t",
        "pdb_path": PDB_DIR / "UBE4B.pdb",
        "variant_col": "variant",
        "target_cols": ["score"],
        "chain_id": "A",
        "auto_guess": 1,
        "span": 3,
    },
    "GRB2": {
        "data_path": EXPERIMENTAL_DIR / "grb2_with_full_sequence.tsv",
        "sep": "\t",
        "pdb_path": PDB_DIR / "GRB2.pdb",
        "variant_col": "variant",
        "target_cols": ["score"],
        "chain_id": "A",
        "auto_guess": 1,
        "span": 3,
    },
    "PTEN_activity": {
        "data_path": EXPERIMENTAL_DIR / "pten-activity_with_full_sequence.tsv",
        "sep": "\t",
        "pdb_path": PDB_DIR / "PTEN.pdb",
        "variant_col": "variant",
        "target_cols": ["score"],
        "chain_id": "A",
        "auto_guess": 1,
        "span": 3,
    },
    "PTEN_abundance": {
        "data_path": EXPERIMENTAL_DIR / "pten-abundance_with_full_sequence.tsv",
        "sep": "\t",
        "pdb_path": PDB_DIR / "PTEN.pdb",
        "variant_col": "variant",
        "target_cols": ["score"],
        "chain_id": "A",
        "auto_guess": 1,
        "span": 3,
    },
    "Alpha-Amylase": {
        "data_path": TOURNAMENT_DIR / "Alpha-Amylase (in silico_ Zero Shot).csv",
        "sep": ",",
        "pdb_path": TOURNAMENT_DIR / "alpha_amylase_wt.pdb",
        "variant_col": "mutant",
        "target_cols": ["expression", "thermostability", "specific activity"],
        "chain_id": "A",
        "auto_guess": 1,
        "span": 3,
    },
    "DLG4_abundance": {
        "data_path": "/Users/kevinbauer/Desktop/Masterarbeit/Zero-Shot_Phase/Final/test_data/dlg4-2022-abundance.tsv",
        "sep": "\t",
        "pdb_path": "/Users/kevinbauer/Desktop/Masterarbeit/Zero-Shot_Phase/Final/test_data/DLG4.pdb",
        "variant_col": "variant",
        "target_cols": ["score"],
        "chain_id": "A",
        "auto_guess": 1,
        "span": 3,
    }, 
    "DLG4_binding": {
        "data_path": "/Users/kevinbauer/Desktop/Masterarbeit/Zero-Shot_Phase/Final/test_data/dlg4-2022-binding.tsv",
        "sep": "\t",
        "pdb_path": "/Users/kevinbauer/Desktop/Masterarbeit/Zero-Shot_Phase/Final/test_data/DLG4.pdb",
        "variant_col": "variant",
        "target_cols": ["score"],
        "chain_id": "A",
        "auto_guess": 1,
        "span": 3,
    },
}

# Rosetta pipeline

In [10]:
# Basic Rosetta and variant helpers
def clean_protein_only(pose_in):
    """Remove non-protein residues such as ligands, ions, water and modifications."""
    pose_out = rosetta.core.pose.Pose()
    pose_out.assign(pose_in)
    rosetta.core.pose.remove_nonprotein_residues(pose_out)
    return pose_out


def load_clean_pose(pdb_path):
    pose = rosetta.core.import_pose.pose_from_file(str(pdb_path))
    return clean_protein_only(pose)


def parse_variant(variant: str):
    """Parse single mutation strings such as A123V."""
    match = VARIANT_RE.match(str(variant).strip().upper())
    if match is None:
        raise ValueError(f"Cannot parse variant: {variant}")
    wt_aa, pos, mut_aa = match.groups()
    return wt_aa, int(pos), mut_aa


def is_single_mutation(variant: str) -> bool:
    return VARIANT_RE.match(str(variant).strip().upper()) is not None


def chain_sequence(pose, chain_id="A"):
    """Return chain sequence and corresponding PyRosetta pose residue indices."""
    pdb_info = pose.pdb_info()
    residue_indices = [
        i for i in range(1, pose.total_residue() + 1)
        if pdb_info.chain(i) == chain_id and pose.residue(i).is_protein()
    ]
    sequence = "".join(pose.residue(i).name1() for i in residue_indices)
    return sequence, residue_indices


def guess_offset_from_variants(pose, variants, chain_id="A", guess=1, span=3, sample_size=100):
    """Infer the sequence offset by maximizing WT residue matches."""
    chain_seq, residue_indices = chain_sequence(pose, chain_id=chain_id)
    n = len(residue_indices)
    if n == 0:
        raise ValueError(f"No protein residues found in chain {chain_id}.")

    parsed_variants = []
    for variant in variants[:sample_size]:
        if is_single_mutation(variant):
            wt_aa, pos, _ = parse_variant(variant)
            parsed_variants.append((wt_aa, pos))

    if not parsed_variants:
        raise ValueError("No valid single mutations found for offset calibration.")

    best_offset, best_hits = None, -1
    for offset in range(guess - span, guess + span + 1):
        hits = 0
        for wt_aa, pos in parsed_variants:
            chain_pos_1based = pos - offset + 1
            if 1 <= chain_pos_1based <= n:
                pose_idx = residue_indices[chain_pos_1based - 1]
                if pose.residue(pose_idx).name1() == wt_aa:
                    hits += 1

        if hits > best_hits:
            best_offset, best_hits = offset, hits

    if best_hits <= 0:
        raise ValueError("Offset calibration failed: 0 WT matches.")

    return best_offset, best_hits


aa_1to3 = {
    "A": "ALA", "C": "CYS", "D": "ASP", "E": "GLU", "F": "PHE",
    "G": "GLY", "H": "HIS", "I": "ILE", "K": "LYS", "L": "LEU",
    "M": "MET", "N": "ASN", "P": "PRO", "Q": "GLN", "R": "ARG",
    "S": "SER", "T": "THR", "V": "VAL", "W": "TRP", "Y": "TYR",
}

# Local repacking/minimization and ΔΔG calculation
def pack_and_minimize(pose_mut, sf, focus_index, repack_radius=10.0, do_pack=True, do_min=True):
    """Locally repack and optionally minimize around the mutation site."""
    focus_selector = rosetta.core.select.residue_selector.ResidueIndexSelector(str(focus_index))

    neighborhood_selector = rosetta.core.select.residue_selector.NeighborhoodResidueSelector()
    neighborhood_selector.set_focus_selector(focus_selector)
    neighborhood_selector.set_distance(repack_radius)
    neighborhood_selector.set_include_focus_in_subset(True)

    if do_pack:
        task_factory = rosetta.core.pack.task.TaskFactory()
        task_factory.push_back(rosetta.core.pack.task.operation.RestrictToRepacking())
        task_factory.push_back(
            rosetta.core.pack.task.operation.OperateOnResidueSubset(
                rosetta.core.pack.task.operation.PreventRepackingRLT(),
                neighborhood_selector,
                True,
            )
        )

        packer = rosetta.protocols.minimization_packing.PackRotamersMover(sf)
        packer.task_factory(task_factory)
        packer.apply(pose_mut)

    if do_min:
        subset = neighborhood_selector.apply(pose_mut)

        movemap = rosetta.core.kinematics.MoveMap()
        movemap.set_bb(False)
        movemap.set_chi(False)
        movemap.set_jump(False)

        for i in range(1, pose_mut.total_residue() + 1):
            if subset[i]:
                movemap.set_bb(i, True)
                movemap.set_chi(i, True)

        min_mover = rosetta.protocols.minimization_packing.MinMover()
        min_mover.movemap(movemap)
        min_mover.score_function(sf)
        min_mover.min_type("dfpmin_armijo_nonmonotone")
        min_mover.tolerance(0.2)
        min_mover.cartesian(True)
        min_mover.max_iter(150)
        min_mover.apply(pose_mut)

def calculate_ddg(
    pose,
    sf,
    variant,
    chain_id="A",
    seq_offset=1,
    do_pack=True,
    do_min=True,
    repack_radius=10.0,
    dump_pdb_dir=None,
):
    """Calculate Rosetta score difference for a single mutation.

    Sign convention follows your previous notebook: ddg = E_wt - E_mut.
    """
    wt_aa, pos, mut_aa = parse_variant(variant)

    chain_seq, residue_indices = chain_sequence(pose, chain_id=chain_id)
    chain_pos_1based = pos - seq_offset + 1

    if not (1 <= chain_pos_1based <= len(residue_indices)):
        raise IndexError(
            f"Variant position {pos} outside chain {chain_id} "
            f"with offset={seq_offset} and chain length={len(residue_indices)}."
        )

    focus_index = residue_indices[chain_pos_1based - 1]
    pdb_aa = pose.residue(focus_index).name1()

    if pdb_aa != wt_aa:
        raise ValueError(
            f"WT mismatch for {variant}: expected {wt_aa} at variant position {pos}, "
            f"but PDB has {pdb_aa} at chain position {chain_pos_1based}."
        )

    if wt_aa == mut_aa:
        return 0.0

    base_pose = rosetta.core.pose.Pose(pose)
    pack_and_minimize(
        base_pose,
        sf,
        focus_index,
        repack_radius=repack_radius,
        do_pack=do_pack,
        do_min=do_min,
    )
    energy_wt = float(sf(base_pose))

    mut_pose = rosetta.core.pose.Pose(base_pose)
    rosetta.protocols.simple_moves.MutateResidue(focus_index, aa_1to3[mut_aa]).apply(mut_pose)

    pack_and_minimize(
        mut_pose,
        sf,
        focus_index,
        repack_radius=repack_radius,
        do_pack=do_pack,
        do_min=do_min,
    )
    energy_mut = float(sf(mut_pose))

    if dump_pdb_dir is not None:
        dump_pdb_dir = Path(dump_pdb_dir)
        dump_pdb_dir.mkdir(parents=True, exist_ok=True)
        mut_pose.dump_pdb(str(dump_pdb_dir / f"{variant}.pdb"))

    return energy_wt - energy_mut

# Load data

In [ ]:
# Dataset loading, scoring and evaluation helpers
def load_dataset(dataset_name, config):
    """Load a dataset, standardize mutation column to `variant`, and keep single mutations only."""
    df = pd.read_csv(config["data_path"], sep=config["sep"])

    # Fallback: if wrong separator produced one large column
    if len(df.columns) == 1:
        fallback_sep = "\t" if config["sep"] == "," else ","
        df = pd.read_csv(config["data_path"], sep=fallback_sep)

    print(f"[{dataset_name}] Columns after loading:")
    print(df.columns.tolist())

    variant_col = config["variant_col"]

    if variant_col not in df.columns:
        raise KeyError(
            f"[{dataset_name}] Expected variant column '{variant_col}' not found. "
            f"Available columns: {df.columns.tolist()}"
        )

    if variant_col != "variant":
        df = df.rename(columns={variant_col: "variant"})

    df = df.loc[df[variant_col].notna()].copy()
    df["variant"] = df["variant"].astype(str).str.strip().str.upper()

    if "num_mutations" in df.columns:
        df = df.loc[df["num_mutations"] == 1].copy()

    df = df.loc[df["variant"].apply(is_single_mutation)].copy()
    df = df.reset_index(drop=True)

    print(f"[{dataset_name}] Loaded {len(df)} single mutations")
    return df


def simple_progress_bar(iterable, total=None, desc="Processing"):
    total = total or len(iterable)
    start = time.time()

    for i, item in enumerate(iterable, 1):
        yield item

        if i % 10 == 0 or i == total:
            elapsed = time.time() - start
            rate = i / elapsed if elapsed > 0 else 0
            remaining = (total - i) / rate if rate > 0 else 0

            sys.stdout.write(
                f"\r{desc}: {i}/{total} ({100 * i / total:.1f}%) "
                f"- {rate:.2f} it/s, ETA {remaining / 60:.1f} min"
            )
            sys.stdout.flush()

    print()


def evaluate_scores(df, target_cols, score_col=SCORE_COL):
    rows = []

    for target_col in target_cols:
        if target_col not in df.columns:
            continue

        true = pd.to_numeric(df[target_col], errors="coerce")
        pred = pd.to_numeric(df[score_col], errors="coerce")
        mask = true.notna() & pred.notna() & np.isfinite(true) & np.isfinite(pred)

        if mask.sum() < 3:
            rows.append({"target": target_col, "n": int(mask.sum()), "pearson": np.nan, "spearman": np.nan})
            continue

        rows.append(
            {
                "target": target_col,
                "n": int(mask.sum()),
                "pearson": pearsonr(true[mask], pred[mask]).statistic,
                "spearman": spearmanr(true[mask], pred[mask]).correlation,
            }
        )

    return pd.DataFrame(rows)


def compute_rosetta_scores_for_dataset(
    dataset_name,
    config,
    sf,
    sample_n=None,
    do_pack=True,
    do_min=True,
    repack_radius=10.0,
    overwrite_mut_pdbs=True,
):
    """Compute Rosetta scores and mutated PDBs for one single-mutant dataset."""
    print(f"\n===== {dataset_name} =====")

    df = load_dataset(dataset_name, config)

    if sample_n is not None and len(df) > sample_n:
        df = df.sample(n=sample_n, random_state=42).copy()
        print(f"[{dataset_name}] Subsampled to {len(df)} rows")

    pose = load_clean_pose(config["pdb_path"])
    chain_id = config.get("chain_id", "A")
    chain_seq, _ = chain_sequence(pose, chain_id=chain_id)
    print(f"[{dataset_name}] PDB chain {chain_id} length: {len(chain_seq)}")
    print(f"[{dataset_name}] First 20 residues: {chain_seq[:20]}")

    variants = df["variant"].dropna().astype(str).tolist()
    seq_offset, hits = guess_offset_from_variants(
        pose,
        variants,
        chain_id=chain_id,
        guess=config.get("auto_guess", 1),
        span=config.get("span", 3),
    )
    print(f"[{dataset_name}] Auto-offset = {seq_offset} | WT matches in sample = {hits}")

    mut_pdb_dir = MUT_PDB_ROOT / dataset_name
    if overwrite_mut_pdbs and mut_pdb_dir.exists():
        for pdb_file in mut_pdb_dir.glob("*.pdb"):
            pdb_file.unlink()
    mut_pdb_dir.mkdir(parents=True, exist_ok=True)

    ddgs = []
    for variant in simple_progress_bar(df["variant"], total=len(df), desc=f"{dataset_name} ΔΔG"):
        try:
            ddg = calculate_ddg(
                pose=pose,
                sf=sf,
                variant=variant,
                chain_id=chain_id,
                seq_offset=seq_offset,
                do_pack=do_pack,
                do_min=do_min,
                repack_radius=repack_radius,
                dump_pdb_dir=mut_pdb_dir,
            )
        except Exception as exc:
            print(f"\n[WARN] {dataset_name} {variant}: {exc}")
            ddg = np.nan

        ddgs.append(ddg)

    df[SCORE_COL] = ddgs

    out_path = OUTPUT_DIR / f"{dataset_name}_with_rosetta_ddg.csv"
    df.to_csv(out_path, index=False)
    print(f"[{dataset_name}] Saved scores: {out_path}")
    print(f"[{dataset_name}] Saved mutated PDBs: {mut_pdb_dir}")
    print(f"[{dataset_name}] NaNs in {SCORE_COL}: {df[SCORE_COL].isna().sum()} / {len(df)}")

    metrics = evaluate_scores(df, target_cols=config.get("target_cols", []), score_col=SCORE_COL)
    if not metrics.empty:
        print(metrics.round(3))

    return df, metrics

# Run

In [ ]:
# Run Rosetta scoring for all requested single-mutant datasets
sf = rosetta.core.scoring.ScoreFunctionFactory.create_score_function("ref2015_cart")

# For a quick test, set SAMPLE_N = 20. For the full run, set SAMPLE_N = None
SAMPLE_N = None
DO_PACK = True
DO_MIN = True
REPACK_RADIUS = 10.0

rosetta_results = {}
rosetta_metrics = []

for dataset_name, config in DATASET_CONFIG.items():
    df_scored, metrics = compute_rosetta_scores_for_dataset(
        dataset_name=dataset_name,
        config=config,
        sf=sf,
        sample_n=SAMPLE_N,
        do_pack=DO_PACK,
        do_min=DO_MIN,
        repack_radius=REPACK_RADIUS,
        overwrite_mut_pdbs=False,
    )

    rosetta_results[dataset_name] = df_scored

    if not metrics.empty:
        metrics.insert(0, "dataset", dataset_name)
        rosetta_metrics.append(metrics)

if rosetta_metrics:
    rosetta_metrics_df = pd.concat(rosetta_metrics, ignore_index=True)
    display(rosetta_metrics_df.round(3))


===== Alpha-Amylase =====
[Alpha-Amylase] Columns after loading:
['mutant', 'mutated_sequence', 'expression', 'thermostability', 'specific activity']
[Alpha-Amylase] Loaded 8075 single mutations
[Alpha-Amylase] PDB chain A length: 425
[Alpha-Amylase] First 20 residues: LTAPSIKSGTILHAWNWSFN
[Alpha-Amylase] Auto-offset = 1 | WT matches in sample = 100
Alpha-Amylase ΔΔG: 8075/8075 (100.0%) - 0.46 it/s, ETA 0.0 minn
[Alpha-Amylase] Saved scores: rosetta_zero_shot_outputs/Alpha-Amylase_with_rosetta_ddg.csv
[Alpha-Amylase] Saved mutated PDBs: rosetta_zero_shot_outputs/mut_pdbs/Alpha-Amylase
[Alpha-Amylase] NaNs in ddg: 0 / 8075
              target     n  pearson  spearman
0         expression  7575    0.147     0.349
1    thermostability  7574    0.026     0.220
2  specific activity  7575    0.085     0.187


,dataset,target,n,pearson,spearman
0,Alpha-Amylase,expression,7575,0.147,0.349
1,Alpha-Amylase,thermostability,7574,0.026,0.220
2,Alpha-Amylase,specific activity,7575,0.085,0.187


In [ ]:
# Optional: add rankings to each Rosetta output file
def add_ranking(
    df,
    score_col=SCORE_COL,
    rank_col="ddg_rank",
    rank_norm_col="ddg_rank_norm",
    higher_is_better=True,
    method="average",
):
    """Add rank and normalized rank. Normalized rank: 0 = best, 1 = worst."""
    df = df.copy()
    scores = pd.to_numeric(df[score_col], errors="coerce")
    mask = scores.notna() & np.isfinite(scores)
    n = int(mask.sum())

    df[rank_col] = np.nan
    df[rank_norm_col] = np.nan

    if n == 0:
        return df

    df.loc[mask, rank_col] = scores[mask].rank(ascending=not higher_is_better, method=method)
    df.loc[mask, rank_norm_col] = 0.0 if n == 1 else (df.loc[mask, rank_col] - 1) / (n - 1)
    return df


for path in sorted(OUTPUT_DIR.glob("*_with_rosetta_ddg.csv")):
    df = pd.read_csv(path)

    if SCORE_COL not in df.columns:
        print(f"Skipping {path.name}: no {SCORE_COL} column")
        continue

    df_ranked = add_ranking(df, score_col=SCORE_COL, higher_is_better=True)
    out_path = path.with_name(path.stem + "_ranked.csv")
    df_ranked.to_csv(out_path, index=False)
    print(f"Saved: {out_path}")

# multi mut_pdbs for FoldVision (supervised)

In [ ]:
# Rosetta workflow
import re
import pandas as pd
import pyrosetta
from pyrosetta import rosetta
from scipy.stats import pearsonr, spearmanr
import os
import sys
import time

pyrosetta.init(
    "-mute all " 
    "-ignore_unrecognized_res true " 
    "-load_PDB_components false "    
    "-linmem_ig 10 "
    "-ex1 -ex2aro -use_input_sc"     
)

def clean_protein_only(pose_in):
    pose_out = rosetta.core.pose.Pose()
    pose_out.assign(pose_in)
    rosetta.core.pose.remove_nonprotein_residues(pose_out)
    return pose_out

# Globale ScoreFunction
sf = rosetta.core.scoring.ScoreFunctionFactory.create_score_function("ref2015_cart")
    
# Parse variant string
def parse_variants(variant: str):
    # akzeptiert: "A123B", "A123B;G45D", "A123B,G45D", "A123B G45D"
    parts = re.split(r"[;, \t]+", variant.strip())
    muts = []
    for p in parts:
        if not p:
            continue
        m = re.match(r"^([A-Z])(\d+)([A-Z])$", p)
        if not m:
            raise ValueError(f"Cannot parse mutation token: {p} (from variant='{variant}')")
        muts.append((m.group(1), int(m.group(2)), m.group(3)))
    if len(muts) == 0:
        raise ValueError(f"No valid mutations in: {variant}")
    return muts


# Because the residue numbering in the PDB and in the TSV may differ, we need to guess the offset.
def guess_offset_from_variants(pose, variants, chain_id="A", guess=0, span=5, sample_size=100):
    """
    Guess sequence offset using all mutation tokens, including multi mutants.
    Example accepted variants:
        A123B
        A123B,G45D
        A123B;G45D
        A123B G45D
    """
    pdb = pose.pdb_info()
    res_indices = [
        i for i in range(1, pose.total_residue() + 1)
        if pdb.chain(i) == chain_id and pose.residue(i).is_protein()
    ]

    n = len(res_indices)
    if n == 0:
        raise ValueError(f"No protein residues found in chain {chain_id}.")

    vlist = []

    for v in variants[:sample_size]:
        try:
            muts = parse_variants(str(v))
            for wt, pos, mt in muts:
                vlist.append((wt, pos))
        except Exception:
            continue

    if not vlist:
        raise ValueError("No valid variants could be parsed for calibration.")

    best_off, best_hits = 0, -1

    for off in range(guess - span, guess + span + 1):
        hits = 0

        for wt, pos in vlist:
            k = pos - off + 1  # 1-basierter Kettenindex

            if 1 <= k <= n:
                i = res_indices[k - 1]

                if pose.residue(i).name1() == wt:
                    hits += 1

        if hits > best_hits:
            best_off, best_hits = off, hits

    if best_hits <= 0:
        raise ValueError("Offset calibration failed (0 WT-matches).")

    return best_off, best_hits

# extract chain sequence from Pose
def chain_sequence(pose, chain_id="A"):
    pdb = pose.pdb_info()
    idxs = [i for i in range(1, pose.total_residue()+1)
            if pdb.chain(i) == chain_id and pose.residue(i).is_protein()]
    seq = "".join(pose.residue(i).name1() for i in idxs)
    return seq, idxs

# 1 letter to 3 letter amino acid code - we need this for Rosetta MutateResidue
aa_1to3 = {
    'A':'ALA','C':'CYS','D':'ASP','E':'GLU','F':'PHE','G':'GLY','H':'HIS','I':'ILE',
    'K':'LYS','L':'LEU','M':'MET','N':'ASN','P':'PRO','Q':'GLN','R':'ARG','S':'SER',
    'T':'THR','V':'VAL','W':'TRP','Y':'TYR'
}

def pack_and_minimize_multi_focus(
    pose_mut,
    sf,
    focus_indices,
    repack_radius=10.0,
    do_pack=True,
    do_min=True
):
    # Select all mutated residues as focus residues
    focus_str = ",".join(str(i) for i in focus_indices)
    sel_focus = rosetta.core.select.residue_selector.ResidueIndexSelector(focus_str)

    # Define neighborhood around all focus residues
    nbh = rosetta.core.select.residue_selector.NeighborhoodResidueSelector()
    nbh.set_focus_selector(sel_focus)
    nbh.set_distance(repack_radius)
    nbh.set_include_focus_in_subset(True)

    # Repack only within neighborhood
    if do_pack:
        tf = rosetta.core.pack.task.TaskFactory()
        tf.push_back(rosetta.core.pack.task.operation.RestrictToRepacking())
        tf.push_back(
            rosetta.core.pack.task.operation.OperateOnResidueSubset(
                rosetta.core.pack.task.operation.PreventRepackingRLT(),
                nbh,
                True
            )
        )

        packer = rosetta.protocols.minimization_packing.PackRotamersMover(sf)
        packer.task_factory(tf)
        packer.apply(pose_mut)

    # Minimize only within neighborhood
    if do_min:
        subset = nbh.apply(pose_mut)

        mm = rosetta.core.kinematics.MoveMap()
        mm.set_bb(False)
        mm.set_chi(False)
        mm.set_jump(False)

        for i in range(1, pose_mut.total_residue() + 1):
            if subset[i]:
                mm.set_bb(i, True)
                mm.set_chi(i, True)

        minmover = rosetta.protocols.minimization_packing.MinMover()
        minmover.movemap(mm)
        minmover.score_function(sf)
        minmover.min_type("dfpmin_armijo_nonmonotone")
        minmover.tolerance(0.2)
        minmover.cartesian(True)
        minmover.max_iter(150)

        minmover.apply(pose_mut)

# Calculate ΔΔG with local repack & min around mutation site
def calculate_ddg_multi(
    pose, sf, variant, chain_id="A", seq_offset=1,
    do_pack=True, do_min=True, repack_radius=10.0, dump_pdp_dir=None
):
    muts = parse_variants(variant)  # list of (wt_aa, pos, mt_aa)

    # WT relax einmal (gemeinsamer Hintergrund)
    base_pose = rosetta.core.pose.Pose(pose)

    # Neighborhood around all mutations 
    chain_seq, res_indices = chain_sequence(base_pose, chain_id)

    focus_idxs = []
    for wt_aa, pos, mt_aa in muts:
        idx_in_chain_1based = pos - seq_offset + 1
        if not (1 <= idx_in_chain_1based <= len(res_indices)):
            raise IndexError(
                f"Variant-position {pos} outside PDB-chain {chain_id} "
                f"(Offset={seq_offset}, Chain-length={len(res_indices)})."
            )
        focus_idx = res_indices[idx_in_chain_1based - 1]

        # WT check
        if base_pose.residue(focus_idx).name1() != wt_aa:
            raise ValueError(
                f"WT-Mismatch for {wt_aa}{pos}{mt_aa}: TSV expects {wt_aa}, "
                f"PDB has {base_pose.residue(focus_idx).name1()} (Offset={seq_offset})."
            )
        focus_idxs.append(focus_idx)

    pack_and_minimize_multi_focus(
        base_pose,
        sf,
        focus_idxs,
        repack_radius=repack_radius,
        do_pack=do_pack,
        do_min=do_min
    )
    E_wt = float(sf(base_pose))

    mut_pose = rosetta.core.pose.Pose(base_pose)

    for wt_aa, pos, mt_aa in muts:
        if wt_aa == mt_aa:
            continue
        idx_in_chain_1based = pos - seq_offset + 1
        focus_idx = res_indices[idx_in_chain_1based - 1]
        rosetta.protocols.simple_moves.MutateResidue(focus_idx, aa_1to3[mt_aa]).apply(mut_pose)

    pack_and_minimize_multi_focus(
        mut_pose,
        sf,
        focus_idxs,
        repack_radius=repack_radius,
        do_pack=do_pack,
        do_min=do_min
    )

    if dump_pdp_dir is not None:
        os.makedirs(dump_pdp_dir, exist_ok=True)
        safe_name = re.sub(r"[^A-Za-z0-9_.-]+", "_", variant)
        mut_pose.dump_pdb(os.path.join(dump_pdp_dir, f"{safe_name}.pdb"))

    E_mut = float(sf(mut_pose))

    return E_wt - E_mut


def simple_progress_bar(iterable, total=None, desc="Processing"):
    total = total or len(iterable)
    start = time.time()
    for i, item in enumerate(iterable, 1):
        yield item
        if i % 10 == 0 or i == total:
            elapsed = time.time() - start
            rate = i / elapsed if elapsed > 0 else 0
            remaining = (total - i) / rate if rate > 0 else 0
            sys.stdout.write(
                f"\r{desc}: {i}/{total} ({100*i/total:.1f}%) "
                f"- {rate:.2f} it/s, ETA {remaining/60:.1f} min"
            )
            sys.stdout.flush()
    print()  # newline at end

# Compute ΔΔG for all variants in a dataframe
def compute_ddg_for_dataframe(
    df, pose, sf, chain_id="A", auto_guess=None, span=5,
    do_pack=True, do_min=True, repack_radius=10.0, compute_pearson=True, compute_spearman=True,
    dump_pdp_dir=None
):
    df = df.copy()

    # calculates sequence offset
    variants = df["variant"].dropna().astype(str).tolist()
    seq_offset, hits = guess_offset_from_variants(
        pose, variants, chain_id=chain_id, guess=auto_guess, span=span
    )
    print(f"[INFO] Auto-Offset = {seq_offset} (WT-Matches in Stichprobe: {hits})")

    # ddg calculation
    ddgs = []
    for v in simple_progress_bar(df["variant"], desc="Computing ΔΔG"):
        try:
            ddgs.append(
                calculate_ddg_multi(
                    pose, sf, v, chain_id=chain_id, seq_offset=seq_offset,
                    do_pack=do_pack, do_min=do_min, repack_radius=repack_radius, dump_pdp_dir=dump_pdp_dir
                )
            )
        except Exception as e:
            print(f"[WARN] {v}: {e}")
            ddgs.append(float("nan"))
    df["ddg"] = ddgs

    if "output" in df.columns:
        tmp = df.dropna(subset=["ddg", "output"])
        if len(tmp) >= 3:
            if compute_pearson:
                r, p = pearsonr(tmp["ddg"], tmp["output"])
                print(f"Pearson r = {r:.3f} (p={p:.1e})")
            if compute_spearman:
                rho, p_s = spearmanr(tmp["ddg"], tmp["output"])
                print(f"Spearman ρ = {rho:.3f} (p={p_s:.1e})")
        else:
            print("Not enough data points to compute correlations (need ≥ 3 after NaN drop).")
    else:
        print("Column 'output' not found; skipping correlations.")

    return df

In [ ]:
import numpy as np
from pathlib import Path

# Generic setup for UBE4B, GRB2, DLG4 abundance and DLG4 binding

PDB_DIR = Path("../../data/experimental_datasets/pdb")

FINAL_DATA_DIR = Path(
    "/Users/kevinbauer/Desktop/Masterarbeit/Supervised Phase/data/Final data"
)

DATASET_CONFIGS = {

    "DLG4_abundance": {
        "base_dir": FINAL_DATA_DIR / "DLG4_abundance" / "multi",
        "pdb_path": "/Users/kevinbauer/Desktop/Masterarbeit/Zero-Shot_Phase/Final/test_data/DLG4.pdb",
        "chain_id": "A",
        "auto_guess": 1,
        "span": 250,
        "mut_pdb_dir": FINAL_DATA_DIR / "DLG4_abundance" / "multi" / "mut_pdbs",
        "out_dir": FINAL_DATA_DIR / "DLG4_abundance" / "multi" / "with_ddg",
    },

    "DLG4_binding": {
        "base_dir": FINAL_DATA_DIR / "DLG4_binding" / "multi",
        "pdb_path": "/Users/kevinbauer/Desktop/Masterarbeit/Zero-Shot_Phase/Final/test_data/DLG4.pdb",
        "chain_id": "A",
        "auto_guess": 1,
        "span": 250,
        "mut_pdb_dir": FINAL_DATA_DIR / "DLG4_binding" / "multi" / "mut_pdbs",
        "out_dir": FINAL_DATA_DIR / "DLG4_binding" / "multi" / "with_ddg",
    },

    "GRB2": {
        "base_dir": FINAL_DATA_DIR / "GRB2" / "multi",
        "pdb_path": PDB_DIR / "GRB2.pdb",
        "chain_id": "A",
        "auto_guess": 1,
        "span": 250,
        "mut_pdb_dir": FINAL_DATA_DIR / "GRB2" / "multi" / "mut_pdbs",
        "out_dir": FINAL_DATA_DIR / "GRB2" / "multi" / "with_ddg",
    },

    "UBE4B": {
        "base_dir": FINAL_DATA_DIR / "UBE4B" / "multi",
        "pdb_path": PDB_DIR / "UBE4B.pdb",
        "chain_id": "A",
        "auto_guess": 1,
        "span": 250,
        "mut_pdb_dir": FINAL_DATA_DIR / "UBE4B" / "multi" / "mut_pdbs",
        "out_dir": FINAL_DATA_DIR / "UBE4B" / "multi" / "with_ddg",
    },
}


def get_existing_splits(base_dir):
    """
    Uses the usual train/val/test files if they exist.
    Missing files are skipped automatically.
    """
    split_paths = {}

    for split_name in ["train", "val", "test"]:
        split_path = base_dir / f"{split_name}.tsv"

        if split_path.exists():
            split_paths[split_name] = split_path
        else:
            print(f"[WARN] Split file not found, skipping: {split_path}")

    if not split_paths:
        raise FileNotFoundError(f"No train/val/test TSV files found in {base_dir}")

    return split_paths


def compute_ddg_for_dataframe_with_fixed_offset(
    df,
    pose,
    sf,
    chain_id="A",
    seq_offset=1,
    do_pack=True,
    do_min=True,
    repack_radius=10.0,
    compute_pearson=True,
    compute_spearman=True,
    dump_pdp_dir=None,
):
    df = df.copy()

    ddgs = []

    for v in simple_progress_bar(df["variant"], desc="Computing ΔΔG"):
        try:
            ddgs.append(
                calculate_ddg_multi(
                    pose,
                    sf,
                    v,
                    chain_id=chain_id,
                    seq_offset=seq_offset,
                    do_pack=do_pack,
                    do_min=do_min,
                    repack_radius=repack_radius,
                    dump_pdp_dir=dump_pdp_dir,
                )
            )
        except Exception as e:
            print(f"[WARN] {v}: {e}")
            ddgs.append(float("nan"))

    df["ddg"] = ddgs

    if "output" in df.columns:
        tmp = df.dropna(subset=["ddg", "output"])

        if len(tmp) >= 3:
            if compute_pearson:
                r, p = pearsonr(tmp["ddg"], tmp["output"])
                print(f"Pearson r = {r:.3f} (p={p:.1e})")

            if compute_spearman:
                rho, p_s = spearmanr(tmp["ddg"], tmp["output"])
                print(f"Spearman ρ = {rho:.3f} (p={p_s:.1e})")
        else:
            print("Not enough data points to compute correlations (need ≥ 3 after NaN drop).")
    else:
        print("Column 'output' not found; skipping correlations.")

    return df


def run_ddg_for_dataset(
    dataset_name,
    cfg,
    sample_n=None,
    do_pack=True,
    do_min=True,
    repack_radius=10.0,
):
    print("\n" + "=" * 70)
    print(f"Running ΔΔG and mutant PDB generation for: {dataset_name}")
    print("=" * 70)

    base_dir = cfg["base_dir"]
    pdb_path = cfg["pdb_path"]
    chain_id = cfg["chain_id"]
    auto_guess = cfg["auto_guess"]
    span = cfg["span"]
    mut_pdb_base = cfg["mut_pdb_dir"]
    out_dir = cfg["out_dir"]

    out_dir.mkdir(parents=True, exist_ok=True)
    mut_pdb_base.mkdir(parents=True, exist_ok=True)

    splits = get_existing_splits(base_dir)

    print(f"[INFO] Base dir: {base_dir}")
    print(f"[INFO] PDB path: {pdb_path}")
    print(f"[INFO] Mutant PDB dir: {mut_pdb_base}")
    print(f"[INFO] Output dir: {out_dir}")

    # Load and clean PDB once per dataset
    pose = rosetta.core.import_pose.pose_from_file(str(pdb_path))
    pose_clean = clean_protein_only(pose)

    # Load all split files
    split_dfs = {}

    for split_name, split_path in splits.items():
        print(f"[INFO] Loading {split_name}: {split_path}")
        df = pd.read_csv(split_path, sep="\t")
        split_dfs[split_name] = df

    # Guess offset once using variants from all available splits
    all_variants = []

    for df in split_dfs.values():
        all_variants.extend(df["variant"].dropna().astype(str).tolist())

    seq_offset, hits = guess_offset_from_variants(
        pose_clean,
        all_variants,
        chain_id=chain_id,
        guess=auto_guess,
        span=span,
        sample_size=min(500, len(all_variants)),
    )

    print(f"[INFO] Auto-Offset = {seq_offset} (WT-Matches in sample: {hits})")

    results = {}

    for split_name, df_full in split_dfs.items():
        print(f"\n--- Processing {dataset_name} split: {split_name} ---")

        df_split = df_full.copy()

        # Optional subsampling for testing
        if sample_n is not None and len(df_split) > sample_n:
            df_split = df_split.sample(n=sample_n, random_state=42)

        split_mut_pdb_dir = mut_pdb_base / split_name
        split_mut_pdb_dir.mkdir(parents=True, exist_ok=True)

        df_split_ddg = compute_ddg_for_dataframe_with_fixed_offset(
            df_split,
            pose_clean,
            sf,
            chain_id=chain_id,
            seq_offset=seq_offset,
            do_pack=do_pack,
            do_min=do_min,
            repack_radius=repack_radius,
            compute_pearson=True,
            compute_spearman=True,
            dump_pdp_dir=str(split_mut_pdb_dir),
        )

        # Merge ddg back into the full split dataframe
        df_full_out = df_full.copy()

        if "ddg" not in df_full_out.columns:
            df_full_out["ddg"] = np.nan

        df_full_out.loc[df_split_ddg.index, "ddg"] = df_split_ddg["ddg"]

        save_path = out_dir / f"{split_name}_with_ddg.tsv"
        df_full_out.to_csv(save_path, sep="\t", index=False)

        print(f"[INFO] Saved {split_name} with ddg to: {save_path}")
        print(f"[INFO] Saved mutant PDBs to: {split_mut_pdb_dir}")

        cols_to_show = ["variant", "ddg"]

        if "output" in df_split_ddg.columns:
            cols_to_show.insert(1, "output")

        print(df_split_ddg[cols_to_show].head())

        results[split_name] = {
            "computed": df_split_ddg,
            "full": df_full_out,
            "save_path": save_path,
            "mut_pdb_dir": split_mut_pdb_dir,
        }

    return results


def run_all_rosetta_mut_pdb_datasets(
    dataset_configs=DATASET_CONFIGS,
    sample_n=None,
    do_pack=True,
    do_min=True,
    repack_radius=10.0,
):
    all_results = {}

    for dataset_name, cfg in dataset_configs.items():
        try:
            results = run_ddg_for_dataset(
                dataset_name=dataset_name,
                cfg=cfg,
                sample_n=sample_n,
                do_pack=do_pack,
                do_min=do_min,
                repack_radius=repack_radius,
            )

            all_results[dataset_name] = results

        except Exception as e:
            print(f"[ERROR] Dataset {dataset_name} failed: {e}")
            all_results[dataset_name] = None

    return all_results


all_rosetta_results = run_all_rosetta_mut_pdb_datasets(
    dataset_configs=DATASET_CONFIGS,
    sample_n=None,
    do_pack=True,
    do_min=True,
    repack_radius=10.0,
)